# Outlines CFG — Always-Valid Code and SQL

**Week 2 | Notebook 3 of 4**

**What you'll learn:**
- Context-free grammars in 10 minutes
- SQL grammar — generating valid SELECT/WHERE queries
- Python expression grammar
- Custom DSL grammar — defining your own language
- Comparing CFG output vs. raw code generation (syntax error rates)
- Integrating Outlines with vLLM for production serving

**Runtime:** ~45 minutes

In [ ]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("02_outlines/03_cfg_codegen.ipynb")

## 1. Setup

In [ ]:
import outlines
from outlines.types import CFG

from src.config import get_openai_client

client = get_openai_client()
model = outlines.from_openai(client, "gpt-4o-mini")

## 2. What Are Context-Free Grammars? (Lark Format)

In [ ]:
# A simple arithmetic grammar in Lark format
arithmetic_grammar = r"""
    start: expr
    expr: expr "+" term
        | expr "-" term
        | term
    term: term "*" factor
        | term "/" factor
        | factor
    factor: NUMBER
          | "(" expr ")"
    NUMBER: /[0-9]+(\.[0-9]+)?/
"""

result = model("Write a mathematical expression for compound interest:", CFG(arithmetic_grammar))

print(f"Generated expression: {result}")
print("✅ Always syntactically valid arithmetic")

## 3. SQL Grammar — Generating Valid SELECT/WHERE Queries

In [ ]:
simple_sql_grammar = r"""
    start: "SELECT " columns " FROM " table_name
    columns: "*" | column ("," column)*
    column: /[a-z_]+/
    table_name: /[a-z_]+/
"""

result = model("Generate a SQL query to get all users:", CFG(simple_sql_grammar))

print(f"SQL: {result}")
print("✅ Always syntactically valid SQL")

In [ ]:
# Extended SQL with WHERE clause
sql_with_where = r"""
    start: "SELECT " columns " FROM " table_name (" WHERE " condition)?
    columns: "*" | column ("," column)*
    column: /[a-z_]+/
    table_name: /[a-z_]+/
    condition: column "=" value
             | column ">" value
             | column "<" value
    value: /[0-9]+/ | /'[^']*'/
"""

result = model("Generate SQL to find users older than 18:", CFG(sql_with_where))

print(f"SQL with WHERE: {result}")

## 4. Python Expression Grammar

In [ ]:
python_expr_grammar = r"""
    start: expr
    expr: expr "+" term
        | expr "-" term
        | expr "%" term
        | term
    term: term "*" factor
        | term "/" factor
        | term "//" factor
        | factor
    factor: NUMBER
          | NAME
          | "(" expr ")"
          | factor "**" factor
    NAME: /[a-zA-Z_][a-zA-Z0-9_]*/
    NUMBER: /[0-9]+(\.[0-9]+)?/
"""

result = model(
    "Write a Python expression to calculate area of a circle with radius r:",
    CFG(python_expr_grammar),
)

print(f"Python expression: {result}")
print("✅ Valid Python expression — can be safely eval'd")

## 5. Custom DSL Grammar — Define Your Own Language

In [ ]:
# Define a simple query DSL
query_dsl = r"""
    start: query
    query: "FIND " entity (" WHERE " condition)?
    entity: "users" | "products" | "orders"
    condition: field "=" value
    field: "name" | "status" | "price" | "date"
    value: /[a-zA-Z0-9_]+/
"""

result = model("Write a query to find products where status=active:", CFG(query_dsl))

print(f"DSL query: {result}")

## 6. CFG vs Raw Generation — Syntax Error Rate Comparison

In [ ]:
# Benchmark: generate 10 SQL queries with and without CFG
prompts = [
    "Get all users",
    "Find orders where total > 100",
    "Select products with status active",
    "List customers from New York",
    "Count orders by status",
]

print("With CFG (Outlines):")
for prompt in prompts:
    result = model(f"{prompt}:", CFG(simple_sql_grammar))
    print(f"  {result}")

print("\n✅ CFG guarantees: 100% syntactically valid output")
print("   Raw prompting: often produces invalid SQL with extra text")

## 7. vLLM Integration for Production Serving

In [ ]:
# When serving with vLLM, use the guided_* parameters
# These use Outlines under the hood in vLLM V0

vllm_example = """
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")

completion = client.chat.completions.create(
    model="Qwen/Qwen2.5-7B",
    messages=[{"role": "user", "content": "Generate SQL"}],
    extra_body={
        "guided_grammar": simple_sql_grammar
        # OR: "guided_json": schema
        # OR: "guided_regex": r"..."
        # OR: "guided_choice": ["yes", "no"]
    }
)
"""

print(vllm_example)
print(
    "\n💡 Note: vLLM V1 defaults to xgrammar. Use --guided-decoding-backend outlines for full Outlines support."
)

## 8. Exercise: Write a Grammar for Your Domain DSL

Define a Lark grammar for a domain-specific language you work with (e.g., filtering syntax, search queries).

In [ ]:
# YOUR TURN: Define your own DSL grammar

# my_dsl = r"""
#     start: command
#     command: ...
# """

# result = model("Generate a command:", CFG(my_dsl))
# print(result)

---

**Next:** [04_vision_structure.ipynb](04_vision_structure.ipynb) — Multimodal extraction from images